In [1]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from PIL import UnidentifiedImageError

# --- Configuration ---
MAIN_DIR = r"C:\Users\lekhn\OneDrive\Desktop\devanagari conjuncts few shot ocr\main"

IMG_SIZE = 64        # Encoder input size (64x64)
EMBEDDING_DIM = 64   # Output feature vector size
NUM_EPISODES = 1000  # Total training episodes
C_WAY = 5            # Classes per episode (fixed)


In [2]:
# --- TensorFlow Setup ---
nll_loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = keras.optimizers.Adam(learning_rate=1e-3)

In [3]:
# Function to calculate Squared Euclidean Distance
def euclidean_distance(a, b):
    a = tf.expand_dims(a, 1)
    b = tf.expand_dims(b, 0)
    distance = tf.reduce_sum(tf.square(a - b), axis=2)
    return distance


In [4]:
def load_and_augment_data(main_dir, img_size):
    """
    Loads images, generates augmented versions (rotations), and reports class validity.
    Returns: (all_data, train_classes, test_classes)
    """
    all_data = {}

    try:
        class_names = [d for d in os.listdir(main_dir) if os.path.isdir(os.path.join(main_dir, d))]
    except FileNotFoundError:
        print(f"Error: Directory not found at {main_dir}. Please check the path.")
        return {}, [], []

    print(f"Found {len(class_names)} potential classes.")

    for class_name in class_names:
        class_path = os.path.join(main_dir, class_name)
        original_images = []

        for img_name in os.listdir(class_path):
            if img_name.lower().endswith(('.jpeg', '.jpg')):
                img_path = os.path.join(class_path, img_name)
                current_img_name = img_name

                try:
                    img = load_img(img_path, target_size=(img_size, img_size), color_mode="grayscale")
                    img_array = img_to_array(img)
                    img_array_rgb = np.repeat(img_array, 3, axis=-1)
                    original_images.append(img_array_rgb / 255.0)

                except UnidentifiedImageError:
                    print(f"ERROR: Skipped file '{current_img_name}' in class '{class_name}'. File is corrupted or not a valid JPEG/image file.")
                except Exception as e:
                    print(f"An unexpected error occurred processing '{current_img_name}' in class '{class_name}': {e}")

        # Reporting and Skipping small classes
        valid_count = len(original_images)
        if valid_count < 9:
            print(f"WARNING: Class '{class_name}' expected 9 originals, found only {valid_count}. Proceeding with found count.")

        if not original_images:
            print(f"Warning: Class '{class_name}' skipped entirely (0 valid images found).")
            continue

        # --- Augmentation (x4) ---
        images_in_class = []
        for img_array in original_images:
            for angle in [0, 1, 2, 3]:
                rotated_img = np.rot90(img_array, k=angle)
                images_in_class.append(rotated_img)

        all_data[class_name] = images_in_class

    # --- Final Class Split ---
    class_names = list(all_data.keys())
    random.shuffle(class_names)
    split_idx = int(0.8 * len(class_names))
    train_classes = class_names[:split_idx]
    test_classes = class_names[split_idx:]

    print(f"\nTotal classes successfully loaded: {len(all_data)}")
    print(f"Train classes for meta-learning: {len(train_classes)}")
    print(f"Test classes for few-shot evaluation: {len(test_classes)}")

    return all_data, train_classes, test_classes


In [5]:
# --- EXECUTE DATA LOADING AND SPLITTING ---
data_dict, train_classes, test_classes = load_and_augment_data(MAIN_DIR, IMG_SIZE)


Found 210 potential classes.
ERROR: Skipped file 'IMG_E3660.JPG' in class 'ण्ठ'. File is corrupted or not a valid JPEG/image file.
ERROR: Skipped file 'IMG_E3663.JPG' in class 'ण्ठ'. File is corrupted or not a valid JPEG/image file.
ERROR: Skipped file 'IMG_E3664.JPG' in class 'ण्ठ'. File is corrupted or not a valid JPEG/image file.
ERROR: Skipped file 'IMG_E3666.JPG' in class 'ण्ठ'. File is corrupted or not a valid JPEG/image file.
ERROR: Skipped file 'IMG_E3678.JPG' in class 'ण्ण'. File is corrupted or not a valid JPEG/image file.
ERROR: Skipped file 'IMG_E3679.JPG' in class 'ण्ण'. File is corrupted or not a valid JPEG/image file.
ERROR: Skipped file 'IMG_E3681.JPG' in class 'ण्ण'. File is corrupted or not a valid JPEG/image file.
ERROR: Skipped file 'IMG_E3683.JPG' in class 'ण्ण'. File is corrupted or not a valid JPEG/image file.
ERROR: Skipped file 'IMG_E3685.JPG' in class 'ण्ण'. File is corrupted or not a valid JPEG/image file.
ERROR: Skipped file 'IMG_E3686.JPG' in class 'ण्ण'. F

In [6]:
def create_encoder(input_shape, embedding_dim):
    """
    Creates a MobileNetV2 encoder, freezing the base and ensuring the Dense head is trainable.
    BatchNorm stays in inference mode by calling with training=False.
    """
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    # Freeze the base model entirely
    base_model.trainable = False
    for layer in base_model.layers:
        layer.trainable = False

    inputs = keras.Input(shape=input_shape)
    # Keep BN in inference mode
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(embedding_dim, activation=None)(x)
    model = Model(inputs, outputs, name="MobileNet_Encoder")

    trainable_vars_count = len(model.trainable_variables)
    print(f"Encoder Trainable Variables: {trainable_vars_count} (expected: 2 for Dense kernel & bias)")
    return model


In [7]:
# --- INSTANTIATE ENCODER ---
encoder = create_encoder((IMG_SIZE, IMG_SIZE, 3), EMBEDDING_DIM)


C:\Users\lekhn\AppData\Local\Temp\ipykernel_19280\3839619887.py:6: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


Encoder Trainable Variables: 2 (expected: 2 for Dense kernel & bias)


In [8]:
def create_episode(data_dict, selected_classes, c_way, k_shot):
    """
    Generates a single episode, prioritizing successful assembly of C_WAY classes
    and ensuring contiguous labeling.
    """
    successful_classes = []

    # Attempt selection using random sampling until C_WAY classes are found
    while len(successful_classes) < c_way:
        if len(selected_classes) == 0:
            return None, None, None

        candidate_name = random.choice(selected_classes)
        if candidate_name not in successful_classes:
            N_TOTAL = len(data_dict[candidate_name])
            if k_shot < N_TOTAL:
                successful_classes.append(candidate_name)

        # Safety break if the available pool is tiny
        if len(selected_classes) < c_way:
            return None, None, None

    # --- Episode Assembly (Using successful classes) ---
    support_set = []
    query_set = []
    query_labels = []

    for class_idx, class_name in enumerate(successful_classes):
        all_augmented_samples = data_dict[class_name]
        random.shuffle(all_augmented_samples)

        # Split: Fixed K-SHOT for Support, ALL REMAINING for Query
        support_images = all_augmented_samples[:k_shot]
        query_images = all_augmented_samples[k_shot:]

        # Collect data
        support_set.extend(support_images)
        query_set.extend(query_images)

        # Assign contiguous label
        query_labels.extend([class_idx] * len(query_images))

    # Final Check: Must have a query set to calculate loss
    if len(query_set) == 0:
        return None, None, None

    # Convert to Tensors
    support_tensor = tf.convert_to_tensor(np.array(support_set), dtype=tf.float32)
    query_tensor = tf.convert_to_tensor(np.array(query_set), dtype=tf.float32)
    query_labels_tensor = tf.convert_to_tensor(np.array(query_labels), dtype=tf.int32)

    return support_tensor, query_tensor, query_labels_tensor


In [9]:
@tf.function
def episodic_train_step(support_images, query_images, query_labels, k_shot):
    """
    Executes one training step: computes prototypes, calculates loss, and applies gradients.
    Encoder forward pass is inside the tape to ensure gradients flow to the Dense head.
    """
    with tf.GradientTape() as tape:
        # Encode inside the tape (keep base BN in inference mode)
        support_embeddings = encoder(tf.cast(support_images, tf.float32), training=False)
        query_embeddings = encoder(tf.cast(query_images, tf.float32), training=False)

        # Infer C_WAY from the support batch length
        # support_images shape: [C_WAY * k_shot, H, W, 3]
        # Reshape with -1 to infer C_WAY
        support_reshaped = tf.reshape(support_embeddings, (-1, k_shot, EMBEDDING_DIM))  # [C_WAY, k_shot, D]
        prototypes = tf.reduce_mean(support_reshaped, axis=1)  # [C_WAY, D]

        # Calculate logits from negative distances
        distances = euclidean_distance(query_embeddings, prototypes)  # [Q, C_WAY]
        logits = -distances

        # Loss
        loss = nll_loss(query_labels, logits)

    # Apply Gradients to encoder's trainable variables (Dense head)
    gradients = tape.gradient(loss, encoder.trainable_variables)
    vars_and_grads = [(g, v) for g, v in zip(gradients, encoder.trainable_variables) if g is not None]

    if not vars_and_grads:
        tf.print("\nCRITICAL ERROR: No gradients computed. Training skipped for this episode.\n")
        return tf.constant(0.0), tf.constant(0.0)

    optimizer.apply_gradients(vars_and_grads)

    # Accuracy
    predictions = tf.argmax(logits, axis=1, output_type=tf.int32)
    accuracy = tf.reduce_mean(tf.cast(tf.equal(predictions, query_labels), tf.float32))

    return loss, accuracy


In [10]:
def train_model(encoder, data_dict, train_classes, test_classes, num_episodes, c_way, k_shot):
    """
    Episodic training and evaluation.
    Critical fix: pass raw images into episodic_train_step, compute embeddings inside the tape.
    """
    print(f"\n--- Starting Training (C={c_way}, K={k_shot}, Episodes={num_episodes}) ---")

    for episode in range(1, num_episodes + 1):
        # --- TRAINING EPISODE GENERATION WITH RETRY ---
        MAX_RETRIES = 50
        S_train, Q_train, Y_train = None, None, None

        for attempt in range(MAX_RETRIES):
            S_train, Q_train, Y_train = create_episode(data_dict, train_classes, c_way, k_shot)
            if S_train is not None:
                break

        if S_train is None:
            continue

        # --- TRAINING STEP (encoder inside tape) ---
        loss, acc = episodic_train_step(S_train, Q_train, Y_train, k_shot)

        if episode % 100 == 0:
            print(f"Episode {episode}/{num_episodes} | Train Loss: {loss.numpy():.4f}, Train Acc: {acc.numpy():.4f}")

    # --- Evaluation Episodes (Testing) ---
    print(f"\nStarting Test Evaluation over 200 episodes...")
    test_accuracies = []
    NUM_TEST_EPISODES = 200

    for test_ep in range(NUM_TEST_EPISODES):
        S_test, Q_test, Y_test = create_episode(data_dict, test_classes, c_way, k_shot)
        if S_test is None:
            continue

        # Compute embeddings for evaluation (no tape)
        S_embed = encoder(tf.cast(S_test, tf.float32), training=False)
        Q_embed = encoder(tf.cast(Q_test, tf.float32), training=False)

        # Infer C_WAY
        num_support = tf.shape(S_test)[0]
        actual_c_way = num_support // k_shot

        # Prototypes and logits
        S_embed_r = tf.reshape(S_embed, (actual_c_way, k_shot, EMBEDDING_DIM))
        prototypes = tf.reduce_mean(S_embed_r, axis=1)  # [C_WAY, D]

        distances = euclidean_distance(Q_embed, prototypes)
        logits = -distances

        loss = nll_loss(Y_test, logits)
        predictions = tf.argmax(logits, axis=1, output_type=tf.int32)
        accuracy = tf.reduce_mean(tf.cast(tf.equal(predictions, Y_test), tf.float32))

        test_accuracies.append(accuracy.numpy())

    mean_test_acc = np.mean(test_accuracies)

    print(f"\n--- Final Test Results (C={c_way}, K={k_shot}) ---")
    print(f"Mean Test Accuracy over {len(test_accuracies)} episodes: {mean_test_acc:.4f}")


In [11]:
# ----------------------------------------------------------------------
#                         FINAL EXECUTION START
# ----------------------------------------------------------------------

# 1. Run 3-Shot Training and Evaluation (Safer initial K-shot)
print("\n--- Running Initial 5-Shot Training ---")
train_model(encoder, data_dict, train_classes, test_classes, NUM_EPISODES, C_WAY, k_shot=5)

# 2. Run 1-Shot Evaluation (using the weights trained in the 3-shot process)
print("\n--- Running 1-Shot Evaluation ---")
train_model(encoder, data_dict, train_classes, test_classes, NUM_EPISODES, c_way=5, k_shot=1)


--- Running Initial 5-Shot Training ---

--- Starting Training (C=5, K=5, Episodes=1000) ---
Episode 100/1000 | Train Loss: 1.3639, Train Acc: 0.7290
Episode 200/1000 | Train Loss: 0.9073, Train Acc: 0.7677
Episode 300/1000 | Train Loss: 0.4172, Train Acc: 0.8707
Episode 400/1000 | Train Loss: 0.9949, Train Acc: 0.6774
Episode 500/1000 | Train Loss: 0.2311, Train Acc: 0.9032
Episode 600/1000 | Train Loss: 0.2266, Train Acc: 0.9355
Episode 700/1000 | Train Loss: 0.6562, Train Acc: 0.7677
Episode 800/1000 | Train Loss: 0.0795, Train Acc: 0.9677
Episode 900/1000 | Train Loss: 0.3739, Train Acc: 0.8581
Episode 1000/1000 | Train Loss: 0.3486, Train Acc: 0.8968

Starting Test Evaluation over 200 episodes...

--- Final Test Results (C=5, K=5) ---
Mean Test Accuracy over 200 episodes: 0.7821

--- Running 1-Shot Evaluation ---

--- Starting Training (C=5, K=1, Episodes=1000) ---
Episode 100/1000 | Train Loss: 7.5282, Train Acc: 0.7371
Episode 200/1000 | Train Loss: 53.0217, Train Acc: 0.5086
E